# 02 · Extracción de Normativas con Docling

Extrae texto de PDFs normativos ecuatorianos usando [Docling](https://github.com/DS4SD/docling) (IBM Research).  
Docling incluye análisis de layout, OCR nativo y detección de tablas,  
lo que lo hace ideal para documentos escaneados o con tablas complejas.

| Parámetro | Opciones |
|-----------|----------|
| `DEVICE` | `cpu` · `mps` (Apple Silicon) · `cuda` (NVIDIA) |
| `OCR_BACKEND` | `auto` · `easyocr` · `mac` |
| `TABLE_MODE` | `accurate` · `fast` |

> **Modelos usados internamente por Docling:**  
> - Layout analysis: `ds4sd/docling-models` (LayoutModel + TableFormer)  
> - OCR: EasyOCR (GPU) o Apple Vision Framework (macOS)  
> Los modelos se descargan automáticamente la primera vez (~1 GB).

## Configuración global

In [1]:
# ── Parámetros de hardware ───────────────────────────────────────────────
DEVICE      = "mps"       # "cpu" | "mps" (Apple Silicon) | "cuda" (NVIDIA)
NUM_THREADS = 8           # hilos CPU (para pre/post-procesamiento)

# ── Parámetros de pipeline ───────────────────────────────────────────────
ENABLE_OCR          = True    # OCR para páginas escaneadas
OCR_BACKEND         = "auto"  # "auto" | "easyocr" | "mac" (Vision Framework)
FORCE_FULL_PAGE_OCR = False   # True para docs 100% escaneados
OCR_CONFIDENCE      = 0.35   # umbral de confianza OCR (0-1)
ENABLE_TABLES       = True    # detectar y extraer tablas
TABLE_MODE          = "accurate"  # "accurate" o "fast"
ENABLE_FORMULAS     = False   # enriquecimiento de fórmulas

from pathlib import Path
DOCS_DIR   = Path("Normativa2026")
OUTPUT_DIR = Path("output/docling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
print(f"{len(pdf_files)} documentos encontrados:")
for f in pdf_files:
    print(f"  {f.name}")


5 documentos encontrados:
  LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988.pdf
  PDL-DERECHOS-DIGITALES.pdf
  Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf
  Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual.pdf
  Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870.pdf


## Sección 1 — Extracción con Docling

Docling usa un pipeline de análisis de documentos con modelos ML:  
1. **Layout analysis** → detecta regiones (texto, tablas, figuras, headers)  
2. **OCR** → transcribe texto en regiones escaneadas  
3. **Table structure** → reconstruye tablas con celdas y spans  
4. **Export** → genera Markdown fiel al layout original

In [2]:
import warnings, platform
warnings.filterwarnings("ignore")

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    AcceleratorOptions,
    AcceleratorDevice,
    EasyOcrOptions,
    OcrMacOptions,
    TableStructureOptions,
)
try:
    from docling.datamodel.pipeline_options import TableFormerMode
except ImportError:
    TableFormerMode = None

# ── Mapeo DEVICE → AcceleratorDevice ─────────────────────────────────────
_DEVICE_MAP = {
    "cpu":  AcceleratorDevice.CPU,
    "mps":  AcceleratorDevice.MPS,
    "cuda": AcceleratorDevice.CUDA,
    "auto": AcceleratorDevice.AUTO,
}
accelerator = _DEVICE_MAP.get(DEVICE, AcceleratorDevice.AUTO)
print(f"Dispositivo: {DEVICE.upper()} → {accelerator}")

# ── OCR backend ───────────────────────────────────────────────────────────
is_macos = platform.system() == "Darwin"
_use_mac_ocr = (OCR_BACKEND == "mac") or (OCR_BACKEND == "auto" and is_macos and DEVICE == "mps")

if ENABLE_OCR:
    if _use_mac_ocr:
        # Requiere ocrmac (Apple Vision Framework)
        try:
            import ocrmac  # noqa: F401
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "ocrmac", "-q"])
        ocr_options = OcrMacOptions(
            lang=["es-ES", "en-US"],
            force_full_page_ocr=FORCE_FULL_PAGE_OCR,
        )
        print("OCR: Apple Vision Framework (OcrMac)")
    else:
        # EasyOCR: soporta GPU (MPS/CUDA); lang usa códigos cortos: "es", "en"
        ocr_options = EasyOcrOptions(
            lang=["es", "en"],
            use_gpu=(DEVICE != "cpu"),
            force_full_page_ocr=FORCE_FULL_PAGE_OCR,
            confidence_threshold=OCR_CONFIDENCE,
        )
        print(f"OCR: EasyOCR (GPU={DEVICE != 'cpu'})")
else:
    ocr_options = None
    print("OCR desactivado")

# ── Opciones de tabla ─────────────────────────────────────────────────────
if ENABLE_TABLES and TableFormerMode is not None:
    # TableFormerMode usa "accurate"/"fast" (strings, no enum members)
    _tmode = TableFormerMode.ACCURATE if TABLE_MODE == "accurate" else TableFormerMode.FAST
    table_options = TableStructureOptions(do_cell_matching=True, mode=_tmode)
elif ENABLE_TABLES:
    table_options = TableStructureOptions(do_cell_matching=True)
else:
    table_options = None

# ── Construir pipeline ────────────────────────────────────────────────────
pipeline_options = PdfPipelineOptions(
    accelerator_options=AcceleratorOptions(
        num_threads=NUM_THREADS,
        device=accelerator,
    ),
    do_ocr=ENABLE_OCR,
    ocr_options=ocr_options,
    do_table_structure=ENABLE_TABLES,
    table_structure_options=table_options,
    do_formula_enrichment=ENABLE_FORMULAS,
    generate_page_images=False,
    generate_picture_images=True,
    images_scale=1.5,   # escala aumentada para mejor OCR en docs de resolución media
)

print(f"\nPipeline configurado:")
print(f"  OCR:    {ENABLE_OCR}  (backend: {'OcrMac' if _use_mac_ocr else 'EasyOCR'})")
print(f"  Tablas: {ENABLE_TABLES}  (modo: {TABLE_MODE})")
print(f"  Threads:{NUM_THREADS}")


Dispositivo: MPS → AcceleratorDevice.MPS
OCR: Apple Vision Framework (OcrMac)

Pipeline configurado:
  OCR:    True  (backend: OcrMac)
  Tablas: True  (modo: accurate)
  Threads:8


In [3]:
# ── Inicializar DocumentConverter ────────────────────────────────────────
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
print("DocumentConverter listo")


DocumentConverter listo


In [4]:
# ── Procesar todos los PDFs ───────────────────────────────────────────────
import time

docs_docling  = {}  # nombre_stem → DoclingDocument
textos_raw    = {}  # nombre_stem → texto markdown

for pdf_path in pdf_files:
    print(f"\n▶  {pdf_path.name}")
    t0 = time.time()
    try:
        result = converter.convert(str(pdf_path))
        doc    = result.document
        texto  = doc.export_to_markdown()

        docs_docling[pdf_path.stem] = doc
        textos_raw[pdf_path.stem]   = texto

        n_pag = len(doc.pages) if doc.pages else 0
        n_tab = len(doc.tables) if doc.tables else 0
        n_fig = len(doc.pictures) if doc.pictures else 0
        elapsed = time.time() - t0

        print(f"  ✓ {n_pag} páginas | {n_tab} tablas | {n_fig} figuras | {len(texto):,} chars | {elapsed:.1f}s")

        # Guardar markdown
        (OUTPUT_DIR / f"{pdf_path.stem}.md").write_text(texto, encoding="utf-8")
        # Guardar JSON nativo de docling
        import json
        try:
            dl_dict = doc.model_dump(mode="json")
        except Exception:
            dl_dict = {}
        with open(OUTPUT_DIR / f"{pdf_path.stem}_docling.json", "w", encoding="utf-8") as jf:
            json.dump(dl_dict, jf, ensure_ascii=False, indent=2)

    except Exception as e:
        docs_docling[pdf_path.stem] = None
        textos_raw[pdf_path.stem]   = ""
        print(f"  ✗ Error: {e}")

print(f"\nTotal procesados: {sum(1 for t in textos_raw.values() if t)}/{len(pdf_files)}")


2026-06-02 15:51:23,798 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-06-02 15:51:23,846 - INFO - Going to convert document batch...
2026-06-02 15:51:23,847 - INFO - Initializing pipeline for StandardPdfPipeline with options hash c916320e66ca0634c14564a6d9a0dcfd



▶  LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988.pdf


2026-06-02 15:51:24,107 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-06-02 15:51:24,107 - INFO - Loading plugin 'docling_defaults'
2026-06-02 15:51:24,108 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-06-02 15:51:24,122 - WARNING - The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
2026-06-02 15:51:24,122 - INFO - Loading plugin 'docling_defaults'
2026-06-02 15:51:24,127 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-06-02 15:51:24,544 - INFO - Accelerator device: 'mps'
2026-06-02 15:51:26,143 - INFO - Accelerator device: 'mps'
2026-06-02 15:51:26,642 - INFO - Processing document LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988.pdf
2026-06-02 15:52:22,178 - INFO - Finished converting document LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEG

  ✓ 104 páginas | 0 tablas | 24 figuras | 201,451 chars | 58.5s

▶  PDL-DERECHOS-DIGITALES.pdf


2026-06-02 15:53:05,649 - INFO - Finished converting document PDL-DERECHOS-DIGITALES.pdf in 43.34 sec.
2026-06-02 15:53:05,745 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-06-02 15:53:05,749 - INFO - Going to convert document batch...
2026-06-02 15:53:05,750 - INFO - Processing document Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Prevenir-el-Lavado-de-Activos-y-la-Financiacion-del-Terrorismo.pdf


...

  ✓ 60 páginas | 3 tablas | 66 figuras | 128,352 chars | 36.8s

▶  Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual.pdf


2026-06-02 15:54:25,102 - INFO - Finished converting document Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual.pdf in 42.53 sec.
2026-06-02 15:54:25,172 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-06-02 15:54:25,177 - INFO - Going to convert document batch...
2026-06-02 15:54:25,177 - INFO - Processing document Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870.pdf


  ✓ 32 páginas | 0 tablas | 44 figuras | 68,032 chars | 42.6s

▶  Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870.pdf


2026-06-02 15:54:39,652 - INFO - Finished converting document Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870.pdf in 14.48 sec.


  ✓ 8 páginas | 0 tablas | 8 figuras | 30,046 chars | 14.5s

Total procesados: 5/5


In [5]:
# ── Explorar estructura nativa de Docling ────────────────────────────────
# Docling modela el documento como elementos tipados (heading, text, table…)
NOMBRE_VER = list(docs_docling.keys())[0]
doc_dl = docs_docling[NOMBRE_VER]

if doc_dl is not None:
    print(f"Documento: {NOMBRE_VER}")
    print(f"Páginas: {len(doc_dl.pages) if doc_dl.pages else 0}")
    print(f"Tablas:  {len(doc_dl.tables) if doc_dl.tables else 0}")
    print(f"Figuras: {len(doc_dl.pictures) if doc_dl.pictures else 0}")

    # Primeros elementos del documento con su label
    print("\n── Primeros 20 elementos del documento ──")
    for i, (el, _) in enumerate(doc_dl.iterate_items()):
        if i >= 20:
            break
        label = getattr(el, "label", "?")
        text  = getattr(el, "text", "") or ""
        print(f"  [{label:20s}] {text[:80]}")

    if doc_dl.tables:
        print("\n── Primera tabla (markdown) ──")
        print(doc_dl.tables[0].export_to_markdown())


Documento: LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988
Páginas: 104
Tablas:  0
Figuras: 24

── Primeros 20 elementos del documento ──
  [picture             ] 
  [text                ] LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGURIDAD
  [section_header      ] Oficio Nro. AN-SG-2026-0305-O
  [section_header      ] Quito, D.M., 19 de mayo de 2026
  [text                ] Asunto: Publicación de la "LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGUR
  [text                ] Señorita Magíster Martha Jaqueline Vargas Camacho Directora del Registro Oficial
  [text                ] De mi consideración:
  [text                ] El Pleno de la Asamblea Nacional, de conformidad con lo previsto en los artículo
  [list_item           ]  Primer debate: 5 de agosto de 2025 (Sesión Nro. 024-AN-2025-2029);
  [list_item           ]  Segundo debate: 10 de febrero de 2026 (Sesión Nro. 069-AN-2025-2029);
  [list_item           ]  Fecha de aprobación: 10 de febrero d

In [6]:
# ── Muestra del markdown extraído ────────────────────────────────────────
texto_muestra = textos_raw[NOMBRE_VER]
print(f"{chr(9552)*70}")
print(f"  DOCUMENTO: {NOMBRE_VER}")
print(f"  Longitud:  {len(texto_muestra):,} caracteres")
print(f"{chr(9552)*70}\n")
print(texto_muestra[:3000])
print("\n[…]")


══════════════════════════════════════════════════════════════════════
  DOCUMENTO: LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGURIDAD_202652616421988
  Longitud:  201,451 caracteres
══════════════════════════════════════════════════════════════════════

<!-- image -->

LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGURIDAD

## Oficio Nro. AN-SG-2026-0305-O

## Quito, D.M., 19 de mayo de 2026

Asunto: Publicación de la "LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGURIDAD"

Señorita Magíster Martha Jaqueline Vargas Camacho Directora del Registro Oficial CORTE CONSTITUCIONAL DEL ECUADOR En su Despacho

De mi consideración:

El Pleno de la Asamblea Nacional, de conformidad con lo previsto en los artículos 120 numeral 6 de la Constitución de la República del Ecuador y 9 numeral 6 de la Ley Orgánica de la Función Legislativa, trató el proyecto de ' LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGURIDAD ' de acuerdo con el siguiente detalle:

1.  Primer debate: 5 de agosto

## Sección 2 — Parser de estructura normativa

El mismo parser de normativas ecuatorianas aplicado al markdown generado por Docling.  
Docling preserva mejor los saltos de línea y el formato de los encabezados,  
lo que mejora la precisión de los patrones regex.

In [7]:
import re
from typing import Optional


# ── Patrones para normativa ecuatoriana ──────────────────────────────────────
_PAT_ART = re.compile(
    r'(?:^|\n)'
    r'(?:#{1,4}[ \t]+|[-*+][ \t]+|\d+\.[ \t]+)?'
    r'[ \t]{0,6}'
    r'Art(?:ículo|iculo|\.)[ \t]+'
    r'(\d+[\w]*)[ \t]*[.\-–—]?[ \t]*'
    r'([^\n]{0,250})',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_JERARQUIA = re.compile(
    r'(?:^|\n)'
    r'(?:#{1,4}[ \t]+)?'
    r'[ \t]{0,4}'
    r'(TÍTULO|CAPÍTULO|SECCIÓN|Título|Capítulo|Sección)[ \t]+'
    r'([IVXLCDM\d]+|PRIMERO|SEGUNDO|TERCERO|CUARTO|QUINTO|SEXTO|'
    r'SÉPTIMO|OCTAVO|NOVENO|DÉCIMO|Único|ÚNICO)[ \t]*[.\-]?[ \t]*\n?'
    r'([^\n]{0,300})',
    re.MULTILINE,
)

_PAT_DISP = re.compile(
    r'(?:^|\n)[ \t]{0,4}'
    r'(DISPOSICIÓN(?:ES)?[ \t]+'
    r'(?:TRANSITORIA|GENERAL|FINAL|DEROGATORIA|REFORMATORIA|SUSTITUTIVA)S?)',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_RESOL = re.compile(
    r'(?:^|\n)[ \t]{0,4}(CONSIDERANDO|RESUELVE|CERTIFICA|DISPONE)[ \t]*:',
    re.MULTILINE,
)

_PAT_ANEXO = re.compile(
    r'(?:^|\n)[ \t]{0,4}'
    r'(ANEXO[ \t]+(?:[IVXLCDM]+|\d+|Único|ÚNICO))[ \t]*[.\-]?[ \t]*\n?'
    r'([^\n]{0,400})',
    re.MULTILINE | re.IGNORECASE,
)

_PAT_FECHA = re.compile(
    r'\b(\d{1,2})\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|'
    r'septiembre|octubre|noviembre|diciembre)\s+de\s+(\d{4})\b',
    re.IGNORECASE,
)


def _tipo_norma(texto: str) -> str:
    s = texto[:800].upper()
    for patron, tipo in [
        (r'RESOLUCIÓN', 'RESOLUCIÓN'),
        (r'PROYECTO DE LEY', 'PROYECTO DE LEY'),
        (r'LEY ORGÁNICA', 'LEY ORGÁNICA'),
        ('LEY', 'LEY'),
    ]:
        if re.search(patron, s):
            return tipo
    return 'NORMATIVA'


def _titulo_norma(texto: str) -> str:
    kws = ('LEY', 'RESOLUCIÓN', 'RESOLUCION', 'PROYECTO', 'REGLAMENTO', 'DECRETO')
    for linea in texto.split('\n')[:30]:
        l = linea.strip()
        if len(l) > 15 and any(k in l.upper() for k in kws):
            return l[:300]
    validas = [l.strip() for l in texto.split('\n') if l.strip()]
    return validas[0][:300] if validas else 'Sin título'


def _fecha_norma(texto: str) -> str:
    '''Extrae la primera fecha del encabezado del documento.'''
    m = _PAT_FECHA.search(texto[:3000])
    if m:
        return f"{m.group(1)} de {m.group(2).lower()} de {m.group(3)}"
    return ""


def _seccion_de_articulo(pos: int, secciones: list) -> str:
    '''Retorna la sección jerárquica más reciente antes de la posición del artículo.'''
    sec = None
    for s in secciones:
        if s['posicion'] <= pos:
            sec = s
        else:
            break
    if sec is None:
        return ""
    parts = [sec['tipo']]
    if sec['identificador']:
        parts.append(sec['identificador'])
    if sec['titulo']:
        parts.append(sec['titulo'][:80])
    return ' '.join(parts)


def parse_normativa(texto: str, archivo: str) -> dict:
    '''Extrae artículos, secciones jerárquicas y anexos de una normativa ecuatoriana.'''
    tipo   = _tipo_norma(texto)
    titulo = _titulo_norma(texto)
    fecha  = _fecha_norma(texto)

    # Artículos (matches crudos)
    am = [(m.group(1), (m.group(2) or '').strip(), m.start(), m.end())
          for m in _PAT_ART.finditer(texto)]

    # Secciones jerárquicas (calculadas antes para asignar a cada artículo)
    secciones = []
    for m in _PAT_JERARQUIA.finditer(texto):
        tit = (m.group(3) or '').strip()
        tit = re.split(r'Art(?:ículo|\.)[ \t]+\d', tit)[0].strip()
        secciones.append({
            'tipo': m.group(1).upper(),
            'identificador': m.group(2).strip(),
            'titulo': tit[:250],
            'posicion': m.start(),
        })
    for m in _PAT_DISP.finditer(texto):
        secciones.append({'tipo': m.group(1).strip().upper(), 'identificador': '',
                          'titulo': '', 'posicion': m.start()})
    for m in _PAT_RESOL.finditer(texto):
        secciones.append({'tipo': m.group(1).upper(), 'identificador': '',
                          'titulo': '', 'posicion': m.start()})
    secciones.sort(key=lambda s: s['posicion'])

    articulos = []
    for i, (num, enc, start, end_m) in enumerate(am):
        next_pos  = am[i + 1][2] if i + 1 < len(am) else min(end_m + 4000, len(texto))
        contenido = re.sub(r'\n{3,}', '\n\n', texto[end_m:next_pos].strip())[:3500]
        seccion   = _seccion_de_articulo(start, secciones)
        articulos.append({
            'numero':     num,
            'encabezado': enc or None,
            'contenido':  contenido,
            'posicion':   start,
            'seccion':    seccion,
            'pagina':     None,   # enriquecido por Docling si aplica
        })

    # Anexos
    axm = list(_PAT_ANEXO.finditer(texto))
    anexos = []
    for i, m in enumerate(axm):
        c_start = m.end()
        c_end   = axm[i + 1].start() if i + 1 < len(axm) else min(c_start + 6000, len(texto))
        contenido = re.sub(r'\n{3,}', '\n\n', texto[c_start:c_end].strip())[:6000]
        anexos.append({
            'identificador': m.group(1).strip().upper(),
            'titulo':        (m.group(2) or '').strip(),
            'contenido':     contenido,
        })

    return {
        'archivo':     archivo,
        'titulo':      titulo,
        'tipo_norma':  tipo,
        'fecha':       fecha,
        'n_articulos': len(articulos),
        'n_secciones': len(secciones),
        'n_anexos':    len(anexos),
        'secciones':   secciones,
        'articulos':   articulos,
        'anexos':      anexos,
    }


In [8]:
# ── Parsear todos los documentos ─────────────────────────────────────────
normativas = {}

for nombre, texto in textos_raw.items():
    if not texto:
        print(f"⚠  {nombre}: texto vacío, omitido")
        continue
    r = parse_normativa(texto, nombre)
    normativas[nombre] = r
    print(
        f"✓ {nombre[:52]:<52}"
        f"  tipo={r['tipo_norma']:<20}"
        f"  arts={r['n_articulos']:>3}"
        f"  secc={r['n_secciones']:>3}"
        f"  anx={r['n_anexos']:>2}"
    )


✓ LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSEGU  tipo=LEY ORGÁNICA          arts= 74  secc=  0  anx= 0
✓ PDL-DERECHOS-DIGITALES                                tipo=PROYECTO DE LEY       arts= 43  secc=  1  anx= 0
✓ Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-Pr  tipo=PROYECTO DE LEY       arts= 93  secc=  1  anx= 0
✓ Proyecto-de-Ley-Transformacion-Digital-y-Audiovisual  tipo=PROYECTO DE LEY       arts=  0  secc=  0  anx= 0
✓ Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870          tipo=NORMATIVA             arts=  1  secc=  0  anx= 0


In [9]:

# ── Visualizar resultados ──────────────────────────────────────────────────────
NOMBRE_VER = list(normativas.keys())[0]  # cambiar por el doc que te interese
doc = normativas[NOMBRE_VER]

print(f"{'═'*70}")
print(f"  {doc['titulo']}")
print(f"  Tipo: {doc['tipo_norma']}   |   {doc['n_articulos']} artículos   |   {doc['n_anexos']} anexos")
print(f"{'═'*70}")

print("\n── Estructura jerárquica ──")
for s in doc['secciones'][:15]:
    ident = f" {s['identificador']}" if s['identificador'] else ''
    print(f"  [{s['tipo']}{ident}]  {s['titulo'][:60]}")

print(f"\n── Primeros artículos ({doc['n_articulos']} total) ──")
for art in doc['articulos'][:5]:
    enc = f"  {art['encabezado']}" if art['encabezado'] else ''
    print(f"\n  Art. {art['numero']}.-{enc}")
    print(f"  {art['contenido'][:350].strip()}")
    print("  …")

if doc['anexos']:
    print(f"\n── Anexos ({doc['n_anexos']}) ──")
    for anx in doc['anexos']:
        print(f"\n  [{anx['identificador']}] {anx['titulo']}")
        print(f"  {anx['contenido'][:200].strip()}")


══════════════════════════════════════════════════════════════════════
  LEY ORGÁNICA PARA EL FORTALECIMIENTO DE LA CIBERSEGURIDAD
  Tipo: LEY ORGÁNICA   |   74 artículos   |   0 anexos
══════════════════════════════════════════════════════════════════════

── Estructura jerárquica ──

── Primeros artículos (74 total) ──

  Art. 20.-  A.-	 Ámbito	 de	 aplicación.	 Las	 disposiciones	 de	 este	 Título	 serán aplicables	a:
  - a) Las	entidades	que	integran	el	sector	público,	conforme	a	lo	previsto	en	el	artículo 225	 de	 la	 Constitución	 de	 la	 República,	 en	 lo	 que	 corresponda	 a	 la	 gestión	 de servicios	esenciales	o	infraestructura	crítica	digital;

Palacio	de	Carondelet	García	Moreno	1043	y	Chile.	Telfs.	3827000

www.presidencia.gob.ec

1

<!-- image -->

##
  …

  Art. 20.-  J.-	 Coordinación	 con	 la	 autoridad	 competente.	 -	 Las	 entidades públicas	 y	 operadores	 de	 infraestructura	 crítica	 digital	 deberán	 designar	 un punto	 de	 contacto	 técnico	 permanente,	 dispon

## Sección 3 — Exportar a Excel

Genera un archivo Excel con una hoja por normativa.
Columnas: `NUMERO` · `ARTICULO` · `PAGINA` · `SECCION` · `FECHA`

In [10]:
# ── Enriquecer artículos con número de página desde provenance de Docling ─
def _build_page_index(doc_dl):
    '''Construye lista (texto[:60], page_no) desde items del documento.'''
    idx = []
    if doc_dl is None:
        return idx
    for item, _ in doc_dl.iterate_items():
        text = (getattr(item, 'text', '') or '').strip()
        if text and hasattr(item, 'prov') and item.prov:
            idx.append((text[:60], item.prov[0].page_no))
    return idx

def _find_page(encabezado, contenido, page_index):
    target = (encabezado or contenido or '')[:40].strip()
    if not target or not page_index:
        return None
    for text_frag, page_no in page_index:
        if target[:20] in text_frag or text_frag[:20] in target:
            return page_no
    return None

for nombre, doc in normativas.items():
    doc_dl   = docs_docling.get(nombre)
    page_idx = _build_page_index(doc_dl)
    for art in doc['articulos']:
        art['pagina'] = _find_page(art.get('encabezado'), art.get('contenido'), page_idx)

print("Páginas asignadas:")
for nombre, doc in normativas.items():
    con_pag = sum(1 for art in doc['articulos'] if art.get('pagina'))
    print(f"  {nombre[:50]}: {con_pag}/{doc['n_articulos']} artículos con página")


Páginas asignadas:
  LEY-ORGANICA-PARA-EL-FORTALECIMIENTO-DE-LA-CIBERSE: 74/74 artículos con página
  PDL-DERECHOS-DIGITALES: 43/43 artículos con página
  Proyecto-de-Ley-Organica-Organica-para-Reprimir-y-: 93/93 artículos con página
  Proyecto-de-Ley-Transformacion-Digital-y-Audiovisu: 0/0 artículos con página
  Resoluci_n_N_SPDP_SPD_2026_0009_R_1771536870: 1/1 artículos con página


In [11]:
import re, json
try:
    import openpyxl
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])
    import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Elimina caracteres de control que XML/Excel no admite
_ILLEGAL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
def _clean(v):
    return _ILLEGAL.sub('', v) if isinstance(v, str) else v

# ── Guardar JSON ──────────────────────────────────────────────────────────
json_path = OUTPUT_DIR / "normativas_docling.json"
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(normativas, f, ensure_ascii=False, indent=2)
print(f"JSON: {json_path}")

# ── Crear Excel ───────────────────────────────────────────────────────────
excel_path = OUTPUT_DIR / "normativas_docling.xlsx"
wb = openpyxl.Workbook()
wb.remove(wb.active)

HDR_FONT  = Font(bold=True, color="FFFFFF", size=11)
HDR_FILL  = PatternFill("solid", fgColor="375623")  # verde oscuro
HDR_ALIGN = Alignment(horizontal="center", vertical="center", wrap_text=True)
CELL_TOP  = Alignment(vertical="top", wrap_text=True)
THIN      = Side(style="thin", color="BFBFBF")
BORDER    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

COLS = [("NUMERO", 10), ("ARTICULO", 90), ("PAGINA", 10), ("SECCION", 45), ("FECHA", 22)]

for nombre, doc in normativas.items():
    ws = wb.create_sheet(title=nombre[:31])

    for col_i, (col_name, col_w) in enumerate(COLS, 1):
        c = ws.cell(row=1, column=col_i, value=col_name)
        c.font, c.fill, c.alignment, c.border = HDR_FONT, HDR_FILL, HDR_ALIGN, BORDER
        ws.column_dimensions[get_column_letter(col_i)].width = col_w
    ws.row_dimensions[1].height = 22
    ws.freeze_panes = "A2"

    fecha = doc.get('fecha', '')
    for row_i, art in enumerate(doc['articulos'], 2):
        texto_art = '\n'.join(filter(None, [art.get('encabezado'), art.get('contenido', '')])).strip()
        vals = [
            art.get('numero', ''),
            texto_art[:32767],
            art.get('pagina') or '',
            art.get('seccion', ''),
            fecha,
        ]
        for col_i, val in enumerate(vals, 1):
            c = ws.cell(row=row_i, column=col_i, value=_clean(val))
            c.alignment = CELL_TOP
            c.border    = BORDER
        ws.row_dimensions[row_i].height = 60

wb.save(excel_path)
print(f"Excel: {excel_path}")
print(f"\n{'Hoja':<35} {'Artículos':>10}  {'Con página':>10}")
print('─' * 58)
for nombre, doc in normativas.items():
    con_pag = sum(1 for art in doc['articulos'] if art.get('pagina'))
    print(f"  {nombre[:33]:<33} {doc['n_articulos']:>10}  {con_pag:>10}")


JSON: output/docling/normativas_docling.json
Excel: output/docling/normativas_docling.xlsx

Hoja                                 Artículos  Con página
──────────────────────────────────────────────────────────
  LEY-ORGANICA-PARA-EL-FORTALECIMIE         74          74
  PDL-DERECHOS-DIGITALES                    43          43
  Proyecto-de-Ley-Organica-Organica         93          93
  Proyecto-de-Ley-Transformacion-Di          0           0
  Resoluci_n_N_SPDP_SPD_2026_0009_R          1           1
